In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score


In [2]:
df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_walmart/WALMART_SALES/walmart_sales.csv")

In [3]:
df = df.dropna(subset=["Date"])
df = df.dropna(subset=["Weekly_Sales"])

In [4]:
df["Store"] = df["Store"].astype(int)
store_counts = df["Store"].value_counts()
print(store_counts)

Store
3     10
14     9
13     9
19     8
1      8
18     7
7      7
5      7
6      6
4      6
8      6
2      6
12     5
17     5
20     4
16     4
9      4
15     3
10     3
11     1
Name: count, dtype: int64


In [5]:
df = df[df["Store"].isin(store_counts[store_counts >= 2].index)]
print(df["Store"].value_counts())

Store
3     10
13     9
14     9
1      8
19     8
5      7
18     7
7      7
2      6
6      6
8      6
4      6
17     5
12     5
20     4
16     4
9      4
15     3
10     3
Name: count, dtype: int64


In [6]:
df['date'] = pd.to_datetime(df['Date'], format = "%d-%m-%Y")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek
df = df.drop(columns=['Date'])

In [7]:
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df["day"] = df["day"].astype(int)
df["weekday"] = df["weekday"].astype(int)

In [8]:
df["Holiday_Flag"] = df["Holiday_Flag"].astype("Int64")

### Split 

In [9]:
X = df.drop(columns=["Weekly_Sales"])
y = df["Weekly_Sales"]

In [10]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "year", "month", "day", "weekday"]
categorical_features = ["Store", "Holiday_Flag"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()) 
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [12]:
cv = KFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [22]:
# Paramètres pour Linear Regression
params_lr = {
    "model__fit_intercept": [True, False],
}

# Paramètres pour Ridge
params_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["auto", "svd", "cholesky", "lsqr"],
    "model__fit_intercept": [True, False],
    "model__max_iter": [1000, 5000, 10000],
}

# Paramètres pour Lasso
params_lasso = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__max_iter": [1000, 5000, 10000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

# Paramètres pour ElasticNet
params_elasticnet = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
    "model__l1_ratio": [0.1, 0.5, 0.9],
    "model__max_iter": [1000, 5000, 10000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

In [14]:
pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge()),
])

pipeline_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso()),
])

pipeline_elasticnet = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ElasticNet()),
])

In [48]:
models = {
    "Linear Regression": (pipeline_lr, params_lr),
    "Ridge": (pipeline_ridge, params_ridge),
    "Lasso": (pipeline_lasso, params_lasso),
    "ElasticNet": (pipeline_elasticnet, params_elasticnet)
}

In [15]:
scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

In [53]:
grid_results = {}

for model_name, (pipeline, params) in models.items():

    print(f"Performing Grid Search for {model_name} : ")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=scoring,
        refit="R2",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(x_train, y_train)
    grid_results[model_name] = grid_search
    
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Best R2 score for {model_name}: {grid_search.best_score_:.4f}")
    # print(f"Cross-validation results for {model_name}:")
    print()
    
    # for score_name in scoring.keys():
    #     mean_score = grid_search.cv_results_[f"mean_test_{score_name}"][grid_search.best_index_]
    #     std_score = grid_search.cv_results_[f"std_test_{score_name}"][grid_search.best_index_]
    #     print(f"{score_name}: {mean_score:.4f} (+/- {std_score:.4f})")

Performing Grid Search for Linear Regression : 
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best parameters for Linear Regression: {'model__fit_intercept': True}
Best R2 score for Linear Regression: 0.9301

Performing Grid Search for Ridge : 
Fitting 5 folds for each of 120 candidates, totalling 600 fits
Best parameters for Ridge: {'model__alpha': 0.01, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}
Best R2 score for Ridge: 0.9309

Performing Grid Search for Lasso : 
Fitting 5 folds for each of 45 candidates, totalling 225 fits
Best parameters for Lasso: {'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}
Best R2 score for Lasso: 0.9316

Performing Grid Search for ElasticNet : 
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.01}
Best R2 score for ElasticNet: 0.9294



In [54]:
def summarize_grid(grid, model_name, metrics):

    results = pd.DataFrame(grid.cv_results_)
    best = results.iloc[grid.best_index_]

    summary = {
        "Model": model_name,
        "Best Params": grid.best_params_
    }

    print(f"\n===== {model_name} =====")
    print(f"Meilleurs paramètres : {grid.best_params_}")

    for metric, sign in metrics.items():

        mean = best[f"mean_test_{metric}"] * sign
        std = best[f"std_test_{metric}"]

        summary[metric] = mean
        summary[f"{metric} std"] = std

        print(f"{metric:<8}: {mean:.4f} (+/- {std:.4f})")

    return (
        summary,
        results.sort_values(f"rank_test_{grid.refit}")
    )

In [59]:
metrics = {
    "R2": 1,
    "MAE": -1,
    "RMSE": -1
}

summary_rows = []
detailed_results = {}

for model_name, grid in grid_results.items():

    summary, details = summarize_grid(
        grid,
        model_name,
        metrics
    )

    summary_rows.append(summary)
    detailed_results[model_name] = details

df_grid = (
    pd.DataFrame(summary_rows)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", None)
display(df_grid.round(4))


===== Linear Regression =====
Meilleurs paramètres : {'model__fit_intercept': True}
R2      : 0.9301 (+/- 0.0164)
MAE     : 136154.3171 (+/- 21803.6550)
RMSE    : 176589.9536 (+/- 28798.4104)

===== Ridge =====
Meilleurs paramètres : {'model__alpha': 0.01, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}
R2      : 0.9309 (+/- 0.0166)
MAE     : 135223.7988 (+/- 23144.4202)
RMSE    : 175598.9704 (+/- 29853.6923)

===== Lasso =====
Meilleurs paramètres : {'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}
R2      : 0.9316 (+/- 0.0164)
MAE     : 134393.4869 (+/- 22983.5347)
RMSE    : 174636.1554 (+/- 29196.6859)

===== ElasticNet =====
Meilleurs paramètres : {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.01}
R2      : 0.9294 (+/- 0.0164)
MAE     : 138101.4266 (+/- 20573.5797)
RMSE    : 177250.7143 (+/- 29440.2596)


,Model,Best Params,R2,R2 std,MAE,MAE std,RMSE,RMSE std
0,Lasso,"{'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}",0.9316,0.0164,134393.4869,22983.5347,174636.1554,29196.6859
1,Ridge,"{'model__alpha': 0.01, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}",0.9309,0.0166,135223.7988,23144.4202,175598.9704,29853.6923
2,Linear Regression,{'model__fit_intercept': True},0.9301,0.0164,136154.3171,21803.6550,176589.9536,28798.4104
3,ElasticNet,"{'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.01}",0.9294,0.0164,138101.4266,20573.5797,177250.7143,29440.2596
